## 4-2 RAG workflow

In [ ]:
# LLM에게 직접 질문
response = llm.invoke([HumanMessage(content=question)])

# 컨텍스트를 포함한 프롬프트 구성
prompt_template = ChatPromptTemplate.from_messages([
	("system", """당신은 AI 온라인 서점의 고객 서비스 상담원입니다.
다음 자료를 참고하여 고객의 질문에 정확하게 답변해주세요.
자료에 없는 내용은 "해당 정보는 제공된 자료에 없습니다."라고 답변하세요.

[참고자료]
{context}"""),
	("human", "{question}")

response = llm.invoke(messages)
print(response.content)

# 키워드 검색 함수
def keyword_search(documents, keyword):
	"""문서 리스트에서 키워드를 포함한 문서 검색"""
	results = []
	for doc in documents:
		if keyword in doc.page_content:
			results.append(doc)
	return results

# Vector store 생성 (문서 단위 - 페이지별)
vectorstore = Chroma.from_documents(
	documents=all_documents,
	embedding=embeddings,
	collection_name="yes24_docs_page"

# Retriever 생성
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}

# 최적 청킹 설정 선택 및 전체 문서 청킹
text_splitter = RecursiveCharacterTextSplitter(
	chunk_size=300,
	chunk_overlap=50,
	length_function=len,
	separators=["\n\n", "\n", ".", " ", ""]

#LangGraph StateGraph로 RAG 파이프라인 구현
# 1. State 타입 정의
class RAGState(TypeDict):
	question: str
	context: str
	answer: str

# 2. retrieve 노드: 질문으로 관련 문서 검색
def retrieve(state: RAGState) -> RAGState:
	"""Vector Store에서 관련 문서를 검색하는 노드"""
	question = state["question"]
	docs = retriever_chunked.invoke(question) # 청킹된 vector store에서 검색
	context = "\n\n".join([doc.page_content for doc in docs]) # 검색 결과를 문자열로 포맷팅
	return {"context" : context}

# 3. generate 노드: 검색된 문서로 답변 생성
def generate(state: RAGstate) -> RAGState:
	"""검색된 문서를 기반으로 답변을 생성하는 노드"""
	question = state["question"]
	context = state["context"]

	# RAG 프롬프트 정의
	rag_prompt = ChatPromptTemplate.from_messages([
		("system", """당신은 AI 온라인 서점 'Yes24'의 고객 서비스 상담원입니다. 다음 검색된 자료를 참고하여 고객의 질문에 정확하고 친절하게 답변해주세요.

[검색된 자료]
{context}

답변 규칙:
1. 검색된 자료를 기반으로 답변하세요
2. 자료에 없는 내용은 추측하지 마세요
3. 존댓말을 사용하세요
4. 간결하고 명확하게 답변하세요"""),
	("human", "{question}")
	])

	# 프롬프트 생성 후 LLM 직접 호출
	messages = rag_prompt.format_message(context=context, question=question)
	response = llm.invoke(messages)
	answer = response.content

	return {"answer": answer}

# 4. StateGraph 구성
# - 노드 추가: retrieve, generate
# - 엣지 연결: START → retrieve → generate → END
workflow = StateGraph(RAGState)

workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# 그래픽 컴파일
rag_graph = workflow.compile()
# 그래프 실행
result = rag_graph.invoke({"question": question})
print(result["answer"])


## 2-2 합성 데이터

In [ ]:
# 환경 변수 설정 및 API 키 로드
load_dotenv(".env")
GMS_KEY = os.getenv("GMS_KEY")

#프롬프팅 기법 비교 실험
#1. zero-shot 프롬프트
zero_shot_prompt = f"{user_query}"
zero_shot_result = chat_completion(zero_shot_prompt, SYSTEM_PROMPT)
print(zero_shot_result)

#2. Few-shot 프롬프트
few_shot_prompt = f"""다음은 영화 추천 예시입니다:

질문: 로맨스 영화 추천해줘
답변: 영화 제목: 노트북
개봉 연도: 2004년
시대를 초월한 순수한 사랑 이야기로, 감동적인 스토리와 아름다운 영상미가 돋보이는 클래식 로맨스 영화입니다.

질문: 코미디 영화를 추천해줘
답변: 영화 제목: 행오버 (The Hangover)
개봉 연도: 2009년
추천 이유: 총각파티 후 기억을 잃은 친구들의 황당한 모험을 그린 코미디로, 예측 불가능한 전개와 유쾌한 유머가 가득합니다.

질문: {user_query}
답변:"""

few_shot_result = chat_completion(few_shot_prompt, SYSTEM_PROMPT)
print(few_shot_result)

# 3. CoT 프롬프트
cot_prompt = f"""{user_query}
영화를 추천하기 전에 다음 단계를 따라 생각해주세요:

1단계: 스릴러 장르의 핵심 요소가 무엇인지 정의합니다 (긴장감, 반전, 서스펜스 등)
2단계: 이 요소들을 잘 갖춘 대표적인 스릴러 영화들을 떠올립니다
3단계: 그 중에서 가장 추천할 만한 영화 1개를 선택하고 이유를 설명합니다

위 단계를 따라 추론 과정을 보여주고, 최종 추천을 해주세요."""

cot_result = chat_completion(cot_prompt, SYSTEM_PROMPT)
print(cot_result)

# 1. 구조화된 출력을 위해 시스템 프롬프트를 작성
STRUCTURED_GENERATOR_SYSTEM_PROMPT = """당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다. 영화는 반드시 하나만 추천합니다.

## 1. 입력 형식
[추천받고자 하는 영화 장르]

## 2. 작업 지시
- 요청된 장르에 가장 적합한 영화 1개를 추천합니다.
- 추천 이유는 구체적이고 설득력 있게 작성합니다.
- 친근하고 유머러스한 말투로 설명합니다.

## 3. 출력 형식
- 출력 형식은 다음 포맷을 따릅니다.
```json
{
    "movie_name": [영화 이름],
    "year": [개봉 연도],
    "genre": [장르],
    "reason": [추천 이유]
}
```
"""

# 1에서 만든프롬프트를 활용하여 답변 받고 답변을 JSON으로 파싱하여 dict 형태로 반환
def generate_movie_recommendation(genre: str, temperature: float = 1.0) ->dict:
    """
    특정 장르에 대한 영화 추천 데이터를 생성

    Args:
        genre: 영화 장르
        temperature: 다양성 조절 파라미터 (0.0~2.0)

    Returns:
        구조화된 영화 추천 데이터
    """
	response = client.chat.completions.create(
		model="gpt-5-mini",
		messages=[
			{"role": "system", "content": STRUCTED_GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": f"{genre} 영화를 추천해줘"}
        ],
        temperature = temperature
    )

    output = response.choices[0].message.content
    return json_parsing(output)

# 3. 3가지 다른 장르에 대해 합성 데이터를 생성하고 syntehtic_data에 저장
genres = ["공포", "SF", "액션"]
synthetic_data = []

for genre in genres:
    print(f"\n{genre} 장르 데이터 생성 중...")
    data = generate_movie_recommendation(genre, temperature=1.0)
    if data:
        synthetic_data.append(data)
        print("생성 완료: ", end=""): pprint(data)

# 평가 결과 분석 및 해석
# 1. 모든 합성 데이터에 대해 평가를 수행하여 `evaluation_results`에 저장해주세요.
evaluation_results = []
for i, data in enumerate(synthetic_data):
    print(f"[{i+1}] {data.get('movie_name', 'Unknown')} 평가 중...")

    data_output = json.dumps(data, ensure_ascii=False)

    # 각 기준에 대해 평가
    scores = []
    for criteria in criteria_list:
        result = evaluate_with_llm(
            instruction=STRUCTURED_GENERATOR_SYSTEM_PROMPT,
            output=data_output,
            criteria=criteria
        )
        if result and "score" in result:
            scores.append(result["score"])

    avg_score = sum(scores) / len(scores) if scores else 0
    evaluation_results.append({
        "data": data,
        "scores": scores,
        "average_score": avg_score
    })
    print(f"   평균 점수: {avg_score:.2f}")

# 2. 평가 결과를 정리하여 출력하세요    
for i, result in enumerate(evaluation_results, 1):
    data = result["data"]
    print(f"\n[{i}] {data.get('movie_name', 'Unknown')} ({data.get('genre', 'Unknown')})")
    print(f"    개별 점수: {result['scores']}")
    print(f"    평균 점수: {result['average_score']:.2f}")

# 2.1 평균 점수와 품질 분석 결과를 출력
total_avg = sum(r["average_score"] for r in evaluation_results) / len(evaluation_results) if evaluation_results else 0

print("품질 분석 결과")
print(f"전체 평균 점수: {total_avg:.2f} / 5.00")

if total_avg >= 4.0:
    print("품질 등급: 우수 - 합성 데이터가 높은 품질을 보입니다.")
elif total_avg >= 3.0:
    print("품질 등급: 보통 - 일부 개선이 필요합니다.")
else:
    print("품질 등급: 미흡 - 프롬프트 또는 생성 파라미터 조정이 필요합니다.")

# 2.2 결과를 `pandas.DataFrame`으로 저장해주세요.
import pandas as pd

df = []
for r in evaluation_results:
    df.append({
        "movie_name": r["data"]["movie_name"],
        "year": r["data"]["year"],
        "genre": r["data"]["genre"],
        "reasons": r["data"]["reason"],
        "Score 1": r["scores"][0],
        "Score 2": r["scores"][1],
        "Score 3": r["scores"][2],
        "Average Score": sum(r["scores"]) / 3,
    })
pd.DataFrame(df)

## 1-1. EDA

In [ ]:
# 데이터셋에 포함된 와인의 총 샘플 수
sample_count = len(df) #len는 행의 수를 센다 

# 데이터셋에 포함된 특성(feature) 수
feature_count = df.shape[1]

# 타깃 변수(y)는 몇 개의 클래스로 구분되는지
class_count = pd.Series(y).nunique()

# 각 클래스별 샘플 수를 Series 형태로 구해서 class_distribution 변수에 담아
class_distribution = pd.Series(y).value_counts().sort_index() 

# Alcohol 평균값이 가장 높은 클래스 번호를 구해
top_alcohol_class = pd.DataFrame({"y": y, "alcohol": df["alcohol"]}).groupby("y")["alcohol"].mean().idxmax() 

# Malic acid 특성의 평균을 구해
malic_mean = df["malic_acid"].mean()

# Malic acid 특성의 표준편차
malic_std = df["malic_acid"].std(ddof=1)

# Color intensity가 10 이상인 샘플의 비율(%)을 구해
high_color_ratio = (df["color_intensity"] >= 10).mean() * 100 

# Ash 특성에서 최소값을 가진 샘플의 클래스
min_ash_idx = df["ash"].idxmin()

# Proline 분포에서 가장 높은 피크를 보이는 클래스 번호를 구해
proline_peak_class = max(peak_by_class, key=peak_by_class.get)

# Magnesium 값이 상위 10%인 샘플들의 평균 Proline 값을 구해
thresh = df["magnesium"].quantile(0.9)
high_magnesium_proline_mean = df.loc[df["magnesium"] >= thresh, "proline"].mean()


# Alcohol과 가장 상관관계가 높은 특성 이름을 구해
top_corr_with_alcohol = df.corr(numeric_only=True)["alcohol"].drop("alcohol")


# 상관관계 시각화 
# 1. 모든 변수들 사이 상관관계를 `corr`이라는 변수에 담아
corr = df.corr()

# 2. 위에서 만든 `corr`의 shape은 정방형태입니다.
# `corr`을 `sns.heatmap` 위에 그려보세요
fig, ax = plt.subplots(figsize=(10, 7))

n = len(corr)
mask = np.triu(np.ones((n, n)))

sns.heatmap(data=corr, annot=True, fmt=".2f", cmap="coolwarm", mask=mask)
ax.grid(False)
ax.set_title("Correlation between features")
ax.set_facecolor("white")
plt.show()


#TODO 3: 다양한 분포 시각화
fig, ax = plt.subplots(figsize=(18, 5), ncols=3)

#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
# 1. 위에서 그린 그림을 세로로 그릴 수도 있을까요?
sns.histplot(data=df, y="flavanoids", bins=20, kde=True, ax=ax[0])
ax[0].set_title('Flavanoids Distribution')

# 2. 클래스별로 flavanoids 어떻게 분포할까요?
sns.histplot(data=df, x="flavanoids", hue="quality", bins=20, kde=True, ax=ax[1])
ax[1].set_title('Flavanoids Distribution across Quality')

# 3. flavanoids와 total_phenols를 동시에 histplot에 그릴 수 있을까요?
sns.histplot(data=df, x="flavanoids", y="total_phenols", hue="quality", bins=20, kde=True, ax=ax[2])
ax[2].set_title('Flavanoids and Total Phenols Distribution')

#★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


# scatterplot을 그려봅시다.
sns.scatterplot(data=df, x="flavanoids", y="total_phenols", hue="quality")
plt.show()

#TODO 5 : pairplot
# 1. 타깃 변수인 `quality`와 상관관계가 가장 높은 5개의 특성을 `top_features`에 할당해주세요.
corr_with_quality = corr["quality"].abs().sort_values(ascending=False)
top_features = corr_with_quality.index[1:6]
print("선택된 feature:", top_features.tolist())

# 2. seaborn.pairplot을 통해 `top_features`에 속하는 특성끼리 scatterplot을 모두 한 군데에 담아주세요.
plot_df = df[top_features.tolist() + ["quality"]]
sns.pairplot(data=plot_df, hue="quality", corner=True)
plt.show()


#TODO 6: 결측치 및 이상치 처리
# 1. 결측치를 평균값으로 대체하여 `df_filled`에 할당해주세요.
df_filled = df_missing.fillna(df_missing.mean(numeric_only=True))

# 2. 이상치를 제거한 데이터를 `df_no_outliers`에 할당해주세요.
df_no_outliers = df_filled[~df_filled.index.isin(outliers_alcohol.index)]


#TODO 7 : sklearn을 통한 데이터 전처
# 1. Train/test 데이터를 나누어 `X_train`, `X_test`, `y_train`, `y_test`에 할당해주세요.

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.3,
                                                    random_state=42,
                                                    stratify=y)


# 2. `StandardScaler`를 사용해 표준화를 진행해주세요.

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

#TODO 8 : sklearn을 통한 모델 훈련 및 검증
clf = LogisticRegression()

# `.fit` 메소드를 통해 모델을 학습시켜주세요
clf.fit(X_train_norm, y_train)

# `X_test`를 예측해서 `y_pred`에 할당해주세요
y_pred = clf.predict(X_test_norm)



#TODO 9 : sklearn을 통한 교차 검증
f1_scores = cross_val_score(estimator=pipe, X=X_train, y=y_train, cv=5, scoring='f1')
f1_mean = f1_scores.mean()   # 요구사항의 '평균 F1-score'



# TODO 10: sklearn을 통한 PCA 분석

X_pca = pca.fit_transform(X_scaled)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y_cluster, palette='Set2', ax=ax[0])
ax[0].set_title("Labels inferred by K-Means")

sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette='Set2', ax=ax[1])
ax[1].set_title("Actual Labels on PCA")

fig.suptitle('KMeans Clustering with PCA')